# DistilBERT Emotion Classification

Fine-tuning `distilbert-base-uncased` on a subset of the `dair-ai/emotion` dataset (1600 train, 400 test).

In [ ]:
%pip install -q transformers datasets accelerate scikit-learn torch

In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

SEED = 42
MODEL_NAME = "distilbert-base-uncased"
DATASET_NAME = "dair-ai/emotion"
TRAIN_SIZE = 1600
TEST_SIZE = 400

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
raw_dataset = load_dataset(DATASET_NAME)
label_names = raw_dataset["train"].features["label"].names
num_labels = len(label_names)
id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in enumerate(label_names)}

train_dataset = raw_dataset["train"].shuffle(seed=SEED).select(range(TRAIN_SIZE))
test_dataset = raw_dataset["test"].shuffle(seed=SEED).select(range(TEST_SIZE))

print(f"Labels ({num_labels}): {label_names}")
print(f"Train samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")
print("Sample row:", train_dataset[0])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_batch, batched=True)
tokenized_test = test_dataset.map(tokenize_batch, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

training_args = TrainingArguments(
    output_dir="emotion_model",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=3e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    seed=SEED,
    logging_steps=25,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
predictions = trainer.predict(tokenized_test)
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=-1)

acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

print(f"Accuracy: {acc:.4f}")
print(f"Macro F1: {macro_f1:.4f}\n")
print(classification_report(y_true, y_pred, target_names=label_names, digits=4))

In [ ]:
sample_texts = [
    "I just got the job offer I have been waiting for all year!",
    "I cannot believe they cancelled the event without telling anyone.",
    "Walking alone down that dark alley made my heart race.",
]

model.eval()
inputs = tokenizer(sample_texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)

for text, prob in zip(sample_texts, probs):
    pred_id = int(torch.argmax(prob).item())
    confidence = float(prob[pred_id].item())
    print(f"Text: {text}")
    print(f"Predicted emotion: {id2label[pred_id]} ({confidence:.4f})\n")